# Análise comparativa de representações
**Fernando Amorim Pontes Neto**

BoW, TF-IDF e dois modelos multilíngues no mesmo corpus. Execute todas as células a partir desta pasta. Os modelos são baixados no primeiro uso e a inferência é local.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from sklearn.metrics.pairwise import cosine_similarity
from ouvidoria_core import *
base = carregar_base('manifestacoes.json')
assert len(base) == 40, 'A análise do desafio requer exatamente 40 registros.'
textos = base.texto.tolist()
ids = base.id.tolist()
saida = Path('resultados'); saida.mkdir(exist_ok=True)
print(f'Base: {len(base)} registros; textos >500 caracteres: {(base.texto.str.len()>500).sum()}')
if Path('PROVENIENCIA.json').exists():
    display(Markdown('**Proveniência:** ' + json.loads(Path('PROVENIENCIA.json').read_text())['descricao']))
display(base.head())

## Representações e pares obrigatórios
BoW e TF-IDF usam a mesma tokenização, minúsculas e remoção de acentos. Mantemos stopwords nesta comparação controlada para evidenciar também a sobreposição de palavras funcionais. O TF-IDF usa IDF suavizada e normalização L2. Nenhum modelo é treinado com os rótulos de categoria. Textos além do limite de tokens são agregados por média de janelas normalizadas, evitando truncamento silencioso.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
bow = CountVectorizer(strip_accents='unicode', lowercase=True)
tfidf = TfidfVectorizer(strip_accents='unicode', lowercase=True)
representacoes = {'BoW': bow.fit_transform(textos), 'TF-IDF': tfidf.fit_transform(textos)}
for nome in MODELOS:
    representacoes[nome.split('/')[-1]] = codificar(textos, nome)
pares = [('M003','M017'), ('M008','M022'), ('M008','M031')]
pos = {ident:i for i,ident in enumerate(ids)}
assert all(a in pos and b in pos for a,b in pares), 'Faltam IDs obrigatórios.'
linhas = []
for a,b in pares:
    r = {'Par': f'{a} × {b}'}
    for nome,x in representacoes.items():
        r[nome] = float(cosine_similarity(x[pos[a]:pos[a]+1],x[pos[b]:pos[b]+1])[0,0])
    linhas.append(r)
tabela = pd.DataFrame(linhas)
display(tabela.round(4))
tabela.to_csv(saida/'comparacao.csv',index=False)
for a,b in pares:
    print(a, textos[pos[a]], '\n', b, textos[pos[b]], '\n')
display(pd.DataFrame([{'Representação':n,'Linhas':x.shape[0],'Dimensões':x.shape[1]} for n,x in representacoes.items()]))

In [ ]:
primeiro = MODELOS[0].split('/')[-1]
segundo = MODELOS[1].split('/')[-1]
display(Markdown(f"Nos pares do enunciado, o primeiro modelo obteve cossenos **{tabela[primeiro].iloc[0]:.3f}**, **{tabela[primeiro].iloc[1]:.3f}** e **{tabela[primeiro].iloc[2]:.3f}**. O segundo obteve **{tabela[segundo].iloc[0]:.3f}**, **{tabela[segundo].iloc[1]:.3f}** e **{tabela[segundo].iloc[2]:.3f}**. Esses valores devem ser interpretados junto aos textos: assunto semelhante não comprova que o local e o evento são os mesmos."))

## Limitações e conclusão
BoW mede coincidência lexical e não reconhece sinônimos sem palavras compartilhadas. TF-IDF reduz a influência dos termos frequentes, mas continua dependente do vocabulário; um endereço raro pode dominar o score. Embeddings aproximam paráfrases, porém podem confundir ocorrências distintas do mesmo tema e produzir escalas diferentes entre modelos. Assim, busca semântica é adequada para recuperar candidatos, enquanto confirmar duplicatas exige verificar local, período e evento. Três pares ilustram diferenças, mas não constituem avaliação estatística da superioridade de um modelo.

## Espaço vetorial e categorias

In [ ]:
e = representacoes[primeiro]
fig, axes = plt.subplots(1,2,figsize=(13,5))
for ax,metodo in zip(axes,['PCA','t-SNE']):
    coords=projetar(e,metodo)
    for categoria in sorted(set(base.categoria_oficial)):
        mask=(base.categoria_oficial==categoria).to_numpy()
        ax.scatter(coords[mask,0],coords[mask,1],label=categoria,alpha=.8)
    for xy,i in zip(coords,ids): ax.annotate(i,xy,fontsize=6)
    ax.set_title(metodo); ax.legend(fontsize=7)
fig.tight_layout(); fig.savefig(saida/'espaco_vetorial.png',dpi=150); plt.show()
sim=e@e.T
mesma=[]; diferente=[]
for i in range(len(base)):
    for j in range(i+1,len(base)):
        (mesma if base.categoria_oficial.iloc[i]==base.categoria_oficial.iloc[j] else diferente).append(sim[i,j])
intra=float(np.mean(mesma)) if mesma else None
inter=float(np.mean(diferente)) if diferente else None
print('Cosseno médio dentro da categoria:',intra,'; entre categorias:',inter)
if intra is not None and inter is not None:
    display(Markdown(f'A média na categoria ({intra:.3f}) é '+('maior' if intra>inter else 'menor ou igual')+f' à média entre categorias ({inter:.3f}). Isso '+('sugere alinhamento temático parcial' if intra>inter else 'não sustenta separação temática por essa medida')+', mas não prova clusters separados. PCA perde variância e t-SNE distorce distâncias globais.'))
(saida/'representacoes_resumo.json').write_text(json.dumps({'pares':linhas,'intra_categoria':intra,'entre_categorias':inter},ensure_ascii=False,indent=2))

## Referências
Materiais fornecidos: lab_representacoes.ipynb e laborat_ecommerce_produtos.ipynb. Implementação: https://www.sbert.net/ e https://scikit-learn.org/stable/modules/feature_extraction.html. O parâmetro stop_words="portuguese" do exemplo não é aceito pelo scikit-learn; por isso não é reproduzido.